## Run me, then select values before running next cell

In [1]:
from src import experiment_widgets

Settings loaded from settings.json


In [5]:
def create_phase_dict(*, phase_widget):
    duration = phase_widget.children[0].children[1].value
    skip_if = phase_widget.children[1].children[1].value
    background_color = phase_widget.children[2].children[1].value  #  ADDED 2/2/2026
    before_instructions = phase_widget.children[3].children[1].value
    during_instructions = phase_widget.children[4].children[1].value
    free_contingency = phase_widget.children[5].children[1].value
    # number_balls is at child index 4 after adding before/during instruction widgets
    number_balls = int(phase_widget.children[6].children[1].value)

    initial_speeds = [phase_widget.children[7].children[i].children[1].value for i in range(number_balls)]
    radii = [phase_widget.children[8].children[i].children[1].value for i in range(number_balls)]
    base_colors = [phase_widget.children[9].children[i].children[1].value for i in range(number_balls)]
    points_per_reinforcement = [phase_widget.children[10].children[i].children[1].value for i in range(number_balls)]
    change_to_clicks = [phase_widget.children[11].children[i].children[1].value for i in range(number_balls)]
    change_over_delay = [phase_widget.children[12].children[i].children[1].value for i in range(number_balls)]
    zero_points_message = [phase_widget.children[13].children[i].children[1].value for i in range(number_balls)]

    # before_instructions = phase_widget.children[2].children[1].value
    # during_instructions = phase_widget.children[3].children[1].value

    # yoked = phase_widget.children[-3].children[1].value
    debug = phase_widget.children[-2].children[1].value

    return {
        "experiment_widgets.duration": duration,

        "experiment_widgets.background_color": background_color,  # ADDED 2/2/2026
        "experiment_widgets.before_instructions": before_instructions,
        "experiment_widgets.during_instructions": during_instructions,
        "experiment_widgets.number_of_balls": number_balls,
        "experiment_widgets.initial_speeds": initial_speeds,
        "experiment_widgets.radii": radii,
        "experiment_widgets.points_per_reinforcement": points_per_reinforcement,
        "experiment_widgets.change_to_clicks": change_to_clicks,
        "experiment_widgets.change_over_delay": change_over_delay,
        "experiment_widgets.base_colors": base_colors,
        "experiment_widgets.time_required": [0.0] * number_balls,
        "experiment_widgets.clicks_required": [1] * number_balls,
        "experiment_widgets.change_from_clicks": [1] * number_balls,
        "experiment_widgets.change_from_delay": [1.0] * number_balls,
        "experiment_widgets.block_score_until_time": [0.0] * number_balls,
        "experiment_widgets.block_score_until_clicks": [0] * number_balls,
        "experiment_widgets.zero_points_message":  zero_points_message,
        # "experiment_widgets.yoked": yoked,
        "experiment_widgets.debug": debug,
        "experiment_widgets.free_contingency": free_contingency,
        "experiment_widgets.skip_if": skip_if,
    }


# Function to gather all phase data into a dictionary with prefixes
def gather_phase_options():
    phase_options = {}
    for phase in range(1, experiment_widgets.phases_input.value + 1):
        phase_key = f"phase_{phase}"
        phase_options[phase_key] = create_phase_dict(phase_widget=experiment_widgets.phase_boxes.children[phase - 1])
    return phase_options


# Auto-load saved settings (no button) so defaults are applied
try:
    experiment_widgets.load_settings(button=None)
except Exception:
    # If load fails, continue with defaults
    pass

# Display the dictionary
print(gather_phase_options())
phase_options = gather_phase_options()

ball_phase_summaries = {}


# %%
import logtocsv
# logtocsv.write_data(string)
# NOTE: Change over delay can be to or from given ball
import pygame
import sys
import os
import numpy as np
from time import strftime  # see format codes: https://docs.python.org/3/library/datetime.html#format-codes
import json

## Define colors here
BLACK = (0, 0, 0)
# LIGHT_BLACK = tuple(min(x + y, 255) for x, y in zip(BLACK, (50, 50, 50))) #This can be used to alter the colors programmatically to be a different color print(LIGHT_BLACK)
RED = (255, 0, 0)
# LIGHT_RED = tuple(min(x + y, 255) for x, y in zip(RED, (50, 50, 50))) #This can be used to alter the colors programmatically to be a different color
DARK_RED = (139, 0, 0)
ORANGE = (255, 165, 0)
DARK_ORANGE = (255, 140, 0)
YELLOW = (255, 255, 0)
DARK_YELLOW = (185, 185, 0)
GREEN = (0, 128, 0)
DARK_GREEN = (0, 100, 0)
BLUE = (0, 0, 255)
DARK_BLUE = (0, 0, 139)
INDIGO = (75, 0, 130)
DARK_INDIGO = (54, 0, 94)
VIOLET = (128, 0, 128)
DARK_VIOLET = (80, 0, 80)
SQUARE_COLOR = (255, 255, 255)
WHITE = (255, 255, 255)
SQUARE_THICKNESS = 4


time_offset = 0
## Define phases here
## Add global blockers based on switching the clicked stimuli
score_clicks_required = 0
last_reinforced_ball = None
last_reinforced_time = None
reinforcement_blocked_until_time = None
# Initialize Pygame
pygame.init()

SCHEDULED_EVENT = pygame.USEREVENT + 1

# pygame.font.init()
font = pygame.font.Font(None, 36)  # Choose a font and size
experimentdate = strftime('%a %d %b %Y, %I:%M%p')
logtocsv.write_data(data=experimentdate)
write_string = 'Scientist:',experiment_widgets.experimenter_input.value,'Participant:',experiment_widgets.participant_input.value
logtocsv.write_data(data=write_string)

os.environ["SDL_VIDEO_CENTERED"] = "1"
clock = pygame.time.Clock()
padding = 0
# For development with multiple monitors:
#surface = pygame.display.set_mode(display=1)
surface = pygame.display.set_mode()
displayX, displayY = surface.get_size()
windowX, windowY = displayX - padding, displayY - padding # Here I was subtracging padding
screen = pygame.display.set_mode((windowX, windowY))  
# screen = pygame.display.set_mode((windowX, windowY), pygame.RESIZABLE,display=1)
pygame.display.set_caption("Resizable Window")

# Set up the square
square_color = (255, 0, 0)
min_margin = 20
square_size = min(windowX, windowY) - 2 * min_margin
square_rect = pygame.Rect((windowX - square_size) // 2, (windowY - square_size) // 2, square_size, square_size)

margin = 100
margin_left = margin
margin_right = margin
margin_top = margin
margin_bottom = margin
values = None

bounce_box_left = margin_left
bounce_box_right = windowX - margin_right
bounce_box_top = windowY - margin_top
square_rect = pygame.Rect((windowX - square_size) // 2, (windowY - square_size) // 2, square_size, square_size)
bounce_box_bottom = margin_bottom

#Random variables for right here:
total_score = 0
current_phase = 1
event = None
current_seconds = 0


## This portion is key for our "Reverse lookup" dictionary
color_names = {
    "BLACK": (0, 0, 0),
    "RED": (255, 0, 0),
    "DARK_RED": (139, 0, 0),
    "ORANGE": (255, 165, 0),
    "DARK_ORANGE": (255, 140, 0),
    "YELLOW": (255, 255, 0),
    "DARK_YELLOW": (185, 185, 0),
    "GREEN": (0, 128, 0),
    "DARK_GREEN": (0, 100, 0),
    "BLUE": (0, 0, 255),
    "DARK_BLUE": (0, 0, 139),
    "INDIGO": (75, 0, 130),
    "DARK_INDIGO": (54, 0, 94),
    "VIOLET": (128, 0, 128),
    "DARK_VIOLET": (80, 0, 80),
    "SQUARE_COLOR": (255, 255, 255),
    "SQUARE_THICKNESS": 4,
}

reverse_lookup = {v: k for k, v in color_names.items()}

scheduled_events = {
    "Event Time": None,
    "Event Type": None,
    "Event Object": None,
    "Status": None,
}

text_rect = None

# Helper render functions

def render_multiline_centered(text, font_obj, color, surface, max_width, y_center=None, y_start=None):
    words = text.split()
    lines = []
    line = ''
    for w in words:
        test = line + (' ' if line else '') + w
        if font_obj.size(test)[0] <= max_width:
            line = test
        else:
            lines.append(line)
            line = w
    if line:
        lines.append(line)
    line_height = font_obj.get_linesize()
    total_h = len(lines) * line_height
    if y_center is None:
        y0 = y_start if y_start is not None else (windowY - total_h) // 2
    else:
        y0 = int(y_center - total_h // 2)
    for i, l in enumerate(lines):
        rendered = font_obj.render(l, True, color)
        rect = rendered.get_rect(center=(windowX // 2, y0 + i * line_height))
        surface.blit(rendered, rect)


def render_top_text(text, font_obj, col, surface, max_width, y_start=20):
    words = text.split()
    lines = []
    line = ''
    for w in words:
        test = line + (' ' if line else '') + w
        if font_obj.size(test)[0] <= max_width:
            line = test
        else:
            lines.append(line)
            line = w
    if line:
        lines.append(line)
    line_height = font_obj.get_linesize()
    for i, l in enumerate(lines):
        rendered = font_obj.render(l, True, col)
        rect = rendered.get_rect(center=(windowX // 2, y_start + i * line_height))
        surface.blit(rendered, rect)



# Function to post a custom event with a timestamp and mouse position
def post_scheduled_event(*, delay, position):
    event_time = pygame.time.get_ticks() + delay
    event = pygame.event.Event(SCHEDULED_EVENT, {
        'timestamp': event_time,
        'position': position
    })
    pygame.event.post(event)




def html_to_rgb(*, html_color):
    if html_color.startswith('#'):
        html_color = html_color[1:]
    r = int(html_color[0:2], 16)
    g = int(html_color[2:4], 16)
    b = int(html_color[4:6], 16)
    return (r, g, b)  


def summarize_phase():
    global sim, phase
    balls_summary= {
    f"ball_{i}": ball.__dict__.copy()
    for i, ball in enumerate(sim.balls, start=1)
    }
    ball_phase_summaries[current_phase] = balls_summary





class Balls:
    # ball = ball(x, y, dx, dy, radius, color, ball_color,clicked_colors[i],reinforcement_interval,change_over_delay)
    def __init__(self, *, x, y, dx, dy, 
                 radius, ball_color, clicked_color, speed,
                 change_to_clicks, change_to_delay,
                 change_from_clicks, change_from_delay,
                 block_score_until_clicks, block_score_until_time,
                 time_required, clicks_required, points_per_reinforcement,
                 zero_points_message
                 ):#fixed_ratio

        self.x = x
        self.y = y
        self.dx = dx
        self.dy = dy
        self.min_speed = speed
        self.max_speed = None
        self.radius = radius
        # self.clicked_color = tuple(int(c * 0.8) for c in self.base_color)
        self.default_color = ball_color
        self.base_color = ball_color
        self.color = ball_color
        self.colorname = reverse_lookup.get(ball_color, str(ball_color))
        self.clicked_color = tuple(int(c * 0.8) for c in self.base_color)
        self.clicked = False
        self.clicks = 0
        self.valid_clicks = 0 # set the amount of clicks to zero, so we can use the fixed ratio & interval
        self.score = 0
        self.block_score_until_time = block_score_until_time
        self.block_score_until_clicks = block_score_until_clicks # self.valid_clicks
        self.change_to_clicks = change_to_clicks_current_ball
        self.change_to_delay = change_to_delay
        self.change_from_clicks = change_from_clicks
        self.change_from_delay = change_from_delay
        self.time_required = time_required
        self.clicks_required = clicks_required
        self.points_per_reinforcement = points_per_reinforcement
        self.zero_points_message = zero_points_message


    def draw(self, screen):
        ## NOTE: Adding a black outline to the balls to make them more visible against the background, especially if the background color is similar to the ball color
        
        pygame.draw.circle(screen, (0, 0, 0), (int(self.x), int(self.y)), self.radius)  # Black outline
        pygame.draw.circle(screen, self.color, (int(self.x), int(self.y)), self.radius - 4)

    def draw_init_position(self,screen):
        self.draw(self,screen)
        print('Draw new ball position')
        pygame.display.flip()

    def advance(self, dt):
        self.x += self.dx * dt
        self.y += self.dy * dt

        if self.x - self.radius < bounce_box_left:
            self.x = bounce_box_left + self.radius
            self.dx = abs(self.dx)
        elif self.x + self.radius > bounce_box_right:
            self.x = bounce_box_right - self.radius
            self.dx = -abs(self.dx)

        if self.y - self.radius < bounce_box_bottom:
            self.y = bounce_box_bottom + self.radius
            self.dy = abs(self.dy)
        elif self.y + self.radius > bounce_box_top:
            self.y = bounce_box_top - self.radius
            self.dy = -abs(self.dy)

    def darken_color(self):
        self.color = tuple(int(c * 0.8) for c in self.base_color)
        #self.color = self.clicked_color# tuple(int(c * 0.8) for c in self.color) #
    def reset_color(self):
        self.color = self.default_color

    # %%
class Simulation:
    def __init__(self, *, phase_options):
        global base_colors, clicked_colors, debug, background_color
        base_colors = phase_options['experiment_widgets.base_colors']
        self.last_reinforced = None
        self.block_score_until_time = 0
        self.last_clicked = None
        self.last_clicked_time = None
        self.last_punishment_time = None
        self.last_reinforcement_time = None  # Must integrate this with the other logic
        self.last_reinforced_ball_click_time = float()  # Added 8/20
        # self.last_points_subtraction = None
        self.change_over_time = None
        self.change_over_clicks = None
        self.reinforcement_texts = []
        self.last_clicked_time = None
        self.last_valid_clicked_time = None

        self.balls = self.init_balls(number_balls=phase_options['experiment_widgets.number_of_balls'],
            initial_speed=phase_options['experiment_widgets.initial_speeds'],
            radii=phase_options['experiment_widgets.radii'],
            base_colors=[html_to_rgb(html_color=color) for color in phase_options['experiment_widgets.base_colors']],  # Generate 'clicked_colors' dynamically\",",
            clicked_colors=[html_to_rgb(html_color=color) for color in phase_options['experiment_widgets.base_colors']],  # Generate 'clicked_colors' dynamically\",",
            change_to_clicks=phase_options['experiment_widgets.change_to_clicks'],
            change_to_delay=phase_options['experiment_widgets.change_over_delay'],
            change_from_clicks=phase_options['experiment_widgets.change_from_clicks'],
            change_from_delay=phase_options['experiment_widgets.change_from_delay'],
            block_score_until_time=phase_options['experiment_widgets.block_score_until_time'],
            block_score_until_clicks=phase_options['experiment_widgets.block_score_until_clicks'],
            time_required=phase_options['experiment_widgets.time_required'],
            clicks_required=phase_options['experiment_widgets.clicks_required'],
            points_per_reinforcement=phase_options['experiment_widgets.points_per_reinforcement'],
            zero_points_message=phase_options['experiment_widgets.zero_points_message']
        )

        debug = phase_options['experiment_widgets.debug']

    def init_balls(self, *, number_balls, initial_speed, radii, base_colors, clicked_colors, 
                   change_to_clicks, change_to_delay, change_from_clicks, change_from_delay, 
                   block_score_until_time, block_score_until_clicks, time_required, 
                   clicks_required, points_per_reinforcement,zero_points_message = ''
                   ):

        global change_to_clicks_current_ball
        print('Init balls')
        balls = []
        logtocsv.write_data(data=('################# INIT balls ######################'))
        event_string = str(current_seconds) + ', Init stimuli, ' + str(total_score) + ', '

        for i in range(int(number_balls)):
            change_to_clicks_current_ball = change_to_clicks[i]
            radius = radii[i]
            speed = initial_speed[i] / 10

            while True:
                x = np.random.uniform(radius + margin + 5, windowX - radius - margin - 5)
                y = np.random.uniform(radius + margin + 5, windowY - radius - margin-5)
                angle = np.random.uniform(0, 2 * np.pi)
                dx = np.random.choice([-1, 1]) * speed * np.cos(angle)
                dy = np.random.choice([-1, 1]) * speed * np.sin(angle)
                color = base_colors[i]

                new_ball = Balls(x=x, y=y, dx=dx, dy=dy, radius=radius, ball_color=base_colors[i], clicked_color=clicked_colors[i],
                                 speed=initial_speed[i], change_to_clicks=change_to_clicks[i], change_to_delay=change_to_delay[i],
                                 change_from_clicks=change_from_clicks[i], change_from_delay=change_from_delay[i],
                                 block_score_until_clicks=block_score_until_clicks[i], block_score_until_time=block_score_until_time[i],
                                 time_required=time_required[i], clicks_required=clicks_required[i], points_per_reinforcement=points_per_reinforcement[i],
                                 zero_points_message = zero_points_message[i]
                                 )

                if not self.check_overlap(new_ball=new_ball, balls=balls):
                    self.append_ball(new_ball=new_ball, balls=balls)

                    self.flip_display()        # Refresh the display
                    break
                else:
                    print('Overlap Detected')
        return balls

    def check_overlap(self, *, new_ball, balls):
        for existing_ball in balls:
            distance = np.hypot(new_ball.x - existing_ball.x, new_ball.y - existing_ball.y)
            if distance < new_ball.radius + existing_ball.radius + 50:  # Adding a margin for safety
                return True
        return False

    def append_ball(self, *, new_ball, balls):
        balls.append(new_ball)

    def flip_display(self):
        # Refresh the display, e.g., using pygame's flip method
        pygame.display.flip()

    def add_reinforcement_text(self, x, y,points_per_reinforcement,font_size = 36):
        if type(points_per_reinforcement) == int:
            if points_per_reinforcement == 0:
                text = ' '
            elif points_per_reinforcement < 0:
                text = str(points_per_reinforcement)
            elif points_per_reinforcement > 0:
                text = '+' + str(points_per_reinforcement)
        else:
            text = str(points_per_reinforcement)
        self.reinforcement_texts.append({
            'text': text,
            'x': x,
            'y': y,
            'font_size': font_size,
            'alpha': 255
        })

    def update_reinforcement_texts(self):
        if self.reinforcement_texts != None:
            for text in self.reinforcement_texts[:]:
                text['y'] -= 1
                text['font_size'] += 1
                text['alpha'] -= 5
                if text['alpha'] <= 0:
                    self.reinforcement_texts.remove(text)

    def draw_reinforcement_texts(self, screen):
        if self.reinforcement_texts is not None:
            for text in self.reinforcement_texts:
                try:
                    font = pygame.font.Font(None, text['font_size'])
                    
                    # Render the outline text
                    outline_color = (0, 0, 0)  # Black outline
                    outline_text = font.render(text['text'], True, outline_color)
                    
                    # Render the main text
                    rendered_text = font.render(text['text'], True, (255, 255, 255))  # White main text
                    rendered_text.set_alpha(text['alpha'])
                    
                    # Draw the outline text by offsetting it in 8 directions
                    x, y = text['x'], text['y']
                    offset = 2  # Thickness of the outline
                    for dx, dy in [(-offset, -offset), (0, -offset), (offset, -offset),
                                (-offset, 0),              (offset, 0),
                                (-offset, offset), (0, offset), (offset, offset)]:
                        screen.blit(outline_text, (x + dx, y + dy))
                    
                    # Draw the main text on top of the outline
                    screen.blit(rendered_text, (x, y))
                    
                    # Update display here if necessary
                except:
                    pass

    def handle_collisions(self):
        for i in range(len(self.balls)):
            for j in range(i + 1, len(self.balls)):
                if np.hypot(self.balls[i].x - self.balls[j].x,
                            self.balls[i].y - self.balls[j].y) < self.balls[i].radius + self.balls[
                    j].radius:
                    self.change_velocities(self.balls[i], self.balls[j])

    def change_velocities(self, p1, p2):
        m1, m2 = p1.radius ** 2, p2.radius ** 2
        M = m1 + m2
        r1, r2 = np.array([p1.x, p1.y]), np.array([p2.x, p2.y])
        # avoid division by zero
        dist_sq = np.linalg.norm(r1 - r2) ** 2
        if dist_sq < 1e-6:
            # jitter positions slightly to separate overlapping balls
            jitter = 0.1
            p1.x += jitter
            p1.y += jitter
            p2.x -= jitter
            p2.y -= jitter
            dist_sq = np.linalg.norm(np.array([p1.x, p1.y]) - np.array([p2.x, p2.y])) ** 2
            if dist_sq < 1e-6:
                return
        d = dist_sq
        v1, v2 = np.array([p1.dx, p1.dy]), np.array([p2.dx, p2.dy])
        u1 = v1 - 2 * m2 / M * np.dot(v1 - v2, r1 - r2) / d * (r1 - r2)
        u2 = v2 - 2 * m1 / M * np.dot(v2 - v1, r2 - r1) / d * (r2 - r1)
        p1.dx, p1.dy = u1
        p2.dx, p2.dy = u2

    def advance(self, dt):
        for ball in self.balls:
            ball.advance(dt)
        self.handle_collisions()

# %%
def main():
    global screen, windowX, windowY, bounce_box_right, bounce_box_top, square_rect, font, text_rect, \
        current_seconds, total_score, phase_duration, current_phase, sim,time_offset
    # callback()
    logtocsv.write_data(data=('################# Phase '+str(current_phase)+' ######################'))
    clock = pygame.time.Clock()
    # sim = Simulation()
    shuffle_button_rect = pygame.Rect(windowX - 150, 20, 120, 30)
    shuffle_button_color = (255, 100, 100)
    # globals()['total_score'] = 0

    
    # while True:
    print(current_seconds)
    for phase in phase_options:
        sim = Simulation(phase_options=phase_options[phase])

        # Read instruction texts
        top_instructions_text = getattr(experiment_widgets, 'top_instructions', None)
        top_instructions = top_instructions_text.value if top_instructions_text is not None else ''
        before_text = phase_options[phase].get('experiment_widgets.before_instructions', '')
        during_text = phase_options[phase].get('experiment_widgets.during_instructions', '')

        # If there are before-phase instructions, display them centered for at least 5s and wait for a click after 5s
        if isinstance(before_text, str) and before_text.strip() and eval(phase_options[phase]["experiment_widgets.skip_if"]) == False:
            
            waiting = True
            start_wait = pygame.time.get_ticks()
            clicked = False
            while waiting:
                for ev in pygame.event.get():
                    if ev.type == pygame.QUIT:
                        pygame.quit()
                        sys.exit()
                    if ev.type == pygame.MOUSEBUTTONDOWN or ev.type == pygame.KEYDOWN:
                        clicked = True
                elapsed = pygame.time.get_ticks() - start_wait
                # Draw instruction screen
                bg_col = html_to_rgb(html_color=phase_options[phase].get('experiment_widgets.background_color', '#000000'))
                # Note: edited this to make text instructions background always black
                screen.fill('#000000')
                # Draw top instructions if present
                if isinstance(top_instructions, str) and top_instructions.strip():
                    render_top_text(top_instructions, font, WHITE, screen, max_width=windowX - 2 * margin, y_start=40)
                # Draw centered before-phase instructions (horiz & vert)
                render_multiline_centered(before_text, font, WHITE, screen, max_width=windowX - 2 * margin)
                pygame.display.flip()
                # If at least 5 seconds have passed and user clicked, continue
                if elapsed >= 5000 and clicked:
                    waiting = False
                    time_offset = elapsed
                clock.tick(30)

        # Proceed with the phase
        print(phase_options[phase]["experiment_widgets.duration"])
        phase_duration = phase_options[phase]["experiment_widgets.duration"]
        

        current_seconds = 0
        end_time = int(phase_duration)
                
        start_time = current_seconds
        # TODO: Implement skip phase logic here!
        if eval(phase_options[phase]["experiment_widgets.skip_if"]):
            print('Should have skipped, see ~line 560')
            print('#######################')
            print('#######################')
            print('#######################')                        
            end_time = current_seconds
            # logtocsv.write_data(f"Skipped Phase after evaluating{phase_options[phase]["experiment_widgets.skip_if"]} as true ")
            #TODO: Implement this feature in our custom phase summaries logging too!


        time_offset = pygame.time.get_ticks()
        while current_seconds < end_time:
            # TODO: Implement free contingency here
            exec(
                phase_options[phase]["experiment_widgets.free_contingency"],
                globals(),
                # locals() ## Can usually not include locals or score doesn't get updated
            )
            # compute real dt in seconds via clock.tick
            dt_ms = clock.tick(60)
            dt = dt_ms / 1000.0
            # Handle events here
            current_seconds = (pygame.time.get_ticks() - time_offset) / 1000
            for event in pygame.event.get():
                event_string = str(current_seconds) + ', ' + str(total_score) + ', '
                if event.type == pygame.QUIT:
                    pygame.quit()
                    sys.exit()

                elif event.type == pygame.KEYDOWN and event.key == pygame.K_s:
                    sim = Simulation(phase_options=phase_options[phase])
                elif event.type == pygame.KEYDOWN and event.key == pygame.K_ESCAPE:
                    pygame.quit()
                    sys.exit()

                elif event.type == SCHEDULED_EVENT:
                    current_ticks = pygame.time.get_ticks()
                    if current_ticks >= event.timestamp:
                        for ball in sim.balls:
                            if ball.clicked:
                                ball.reset_color()
                                ball.clicked = False
                    else:
                        pygame.event.post(pygame.event.Event(SCHEDULED_EVENT, {
                            'timestamp': event.timestamp,
                            'position': event.position
                        }))

                elif event.type == pygame.MOUSEBUTTONDOWN:
                    ## NOTE: Added this here to track the last clicked time globally, regardless of if they clicked on a ball or not.  it just makes sense to have this variable updated on any click, even if it's not on a ball.  We can always add another variable for last valid click time if we want to track that separately for implementing the changeover delay based on valid clicks.
                    sim.last_clicked_time = current_seconds
                    if event.button == 1:
                        #TODO: add logic here to update the last clicked time! IF we want it globally and not even for a ball

                        for ball in sim.balls:
                            if np.hypot(event.pos[0] - ball.x, event.pos[1] - ball.y) < ball.radius: # Handle the clicked ball
                                #TODO: add logic here to update the last clicked time!
                                ## NOTE: This is where we would add it if we wanted it to only update the last clicked time if it was on a ball

                                ball.darken_color()
                                ball.clicked = True
                                ball.clicks += 1
                                post_scheduled_event(delay=500, position='TEST LINE')
                                event_string += "Clicked: "+ str(ball.colorname) + ', '
                                event_string += 'x='+ str(event.pos[0])+', ' + ' y=' + str(event.pos[1]) + ', '

                                #TODO: Here is where we'll check to make sure we're implementing the change over delay correctly
                                #NOTE: variables are:
                                # sim.last_reinforced = ball.colorname
                                # sim.last_clicked == ball:
                                # ball.change_to_delay
                                # sim.last_reinforced_ball_click_time
                                # ball.block_score_until_time
                                # current_seconds
                                # total_score


                                if sim.last_reinforced != ball.colorname and sim.last_reinforced != None:
                                    ## NOTE: Changeover has occured here
                                    ## TODO: revisit all of this code, it can be problematic!  
                                    # Remember: Unintended change over
                                    if sim.last_clicked == ball:
                                        ball.clicks_required -= 1
                                    else:
                                        ball.clicks_required = ball.change_to_clicks - 1
                                        ball.block_score_until_time = ball.change_to_delay + sim.last_reinforced_ball_click_time
                                        sim.last_clicked = ball

                                    if ball.clicks_required <=0:
                                        if current_seconds < ball.block_score_until_time:
                                            break

                                        ball.score += ball.points_per_reinforcement
                                        #TODO: Here is the place that was making the issue!  This should not be called on regular clicks
                                        #TODO: merge this with the TODO: that is like 21 lines under this one!
                                        # sim.add_reinforcement_text(event.pos[0], event.pos[1], ball.points_per_reinforcement)
                                        if isinstance(ball.points_per_reinforcement, int) and ball.points_per_reinforcement != 0:
                                            sim.add_reinforcement_text(event.pos[0], event.pos[1], ball.points_per_reinforcement)
                                        else:
                                            sim.add_reinforcement_text(event.pos[0], event.pos[1], ball.zero_points_message)
                                        ## NOTE: This is where I do the last reward time or last click time when a ball was actually clicked
                                        if ball.points_per_reinforcement > 0:
                                            sim.last_reinforced_ball_click_time = current_seconds
                                            sim.last_reinforced = ball.colorname
                                        total_score += ball.points_per_reinforcement
                                        ball.valid_clicks += 1

                                    if current_seconds < ball.block_score_until_time or current_seconds < sim.block_score_until_time:
                                        break

                                    elif sim.last_reinforced is not None and sim.last_reinforced != ball.colorname:
                                        ## NOTE: Here, I need to carefully fix this statement!
                                        Simulation.last_clicked = ball

                                    elif ball.valid_clicks < ball.block_score_until_clicks:
                                        break

                                else:
                                    ball.score += ball.points_per_reinforcement
                                    #TODO: add ball message here instead for if there's no points per reinforcement
                                    if ball.points_per_reinforcement != 0:
                                        sim.add_reinforcement_text(event.pos[0], event.pos[1], ball.points_per_reinforcement)
                                    else:
                                        sim.add_reinforcement_text(event.pos[0], event.pos[1], ball.zero_points_message)
                                    sim.last_reinforced = ball.colorname
                                    sim.last_reinforced_ball_click_time = current_seconds
                                    sim.last_reinforcement_time = current_seconds                                    
                                    total_score += ball.points_per_reinforcement
                                    ball.block_score_until_time = current_seconds + ball.time_required

                                    for b in sim.balls:
                                        if np.hypot(event.pos[0] - b.x, event.pos[1] - b.y) < b.radius:
                                            pass
                                        else:
                                            b.block_score_until_time = current_seconds + b.time_required

                            elif not shuffle_button_rect.collidepoint(event.pos):
                                event_string += 'Clicked: None, '
                                event_string += f'x={event.pos[0]}, y={event.pos[1]}, '

                    if shuffle_button_rect.collidepoint(event.pos):
                        sim = Simulation(phase_options=phase_options[phase])  # Create a new simulation to reorient all balls
                        event_string += 'Clicked: Shuffle, '
                        event_string += f'x={event.pos[0]}, y={event.pos[1]}, '

                    for ball in sim.balls:
                        color = reverse_lookup.get(ball.color, "Unknown Color")
                        event_string += ' ' + str(color) +':'
                        event_string += ' x='+ str(int(ball.x)) +', '+ ' y='+ str(int(ball.y))+', ' + ' dx='+ str((ball.dx))+ ', '+' dy='+ str((ball.dy))  +', '+' clicks='+ str((ball.clicks))+', '+' score='+ str((ball.score))+','

                    logtocsv.write_data(data=event_string)

                elif event.type == pygame.MOUSEBUTTONUP:
                    if event.button == 1:
                        for ball in sim.balls:
                            if ball.clicked:
                                ball.reset_color()
                                ball.clicked = False
                elif event.type == pygame.VIDEORESIZE:
                    windowX, windowY = event.w, event.h
                    screen = pygame.display.set_mode((windowX, windowY), pygame.RESIZABLE)
                    bounce_box_right = windowX - margin_right
                    bounce_box_top = windowY - margin_top
                    square_rect = pygame.Rect((windowX - square_size) // 2, (windowY - square_size) // 2, square_size,
                                            square_size)

            # Draw phase frame
            # Use background color from phase options if present
            bg_hex = phase_options[phase].get('experiment_widgets.background_color', '#000000')
            try:
                bg_col = html_to_rgb(html_color=bg_hex)
            except Exception:
                bg_col = BLACK
            screen.fill(bg_col)
            sim.advance(dt)

            for ball in sim.balls:
                ball.draw(screen)

            # Always render global top instructions if present
            if isinstance(top_instructions, str) and top_instructions.strip():
                try:
                    render_top_text(top_instructions, font, WHITE, screen, max_width=windowX - 2 * margin, y_start=10)
                except Exception:
                    pass

            # Draw during-phase instructions at top if present
            if isinstance(during_text, str) and during_text.strip():
                render_top_text(during_text, font, WHITE, screen, max_width=windowX - 2 * margin, y_start=40)

            ## TODO: Implement the optional operant rule here

            pygame.draw.rect(screen, SQUARE_COLOR, (margin, margin, windowX - 2 * (margin), windowY - 2 * (margin)),
                            SQUARE_THICKNESS)
            pygame.draw.rect(screen, shuffle_button_color, shuffle_button_rect)
            text_score = font.render(f'Score: {globals()['total_score']}', True, YELLOW)
            text_rect_score = text_score.get_rect(center=(windowX // 2, windowY - 60))
            
            #NEW: 9-3-24
            sim.update_reinforcement_texts()
            sim.draw_reinforcement_texts(screen)

            screen.blit(text_score, text_rect_score)

            if debug:
                debug_info = [
                    sim.balls[1].clicks,
                    "Phase:"+phase,
                    "Phase Duration: "+str(phase_options[phase]["experiment_widgets.duration"]),
                    'end time:'+str(end_time),
                    "Time Remaining:"+str(round(end_time - current_seconds, 1)),
                    "Current Time"+str(round(current_seconds, 1)),
                    "last reinforced:"+str(sim.last_reinforced),
                    "scoring blocked until:" + str(ball.block_score_until_time)
                ]

                # Starting y-position for the text
                start_y = 90
                line_height = 35  # Adjust this according to your font size and spacing
                for i, attributes in enumerate(debug_info):
                    text = font.render(f'{attributes}', True, YELLOW)
                    text_rect = text.get_rect(center=(windowX - 250, start_y + i * line_height))
                    screen.blit(text, text_rect)


            font = pygame.font.Font(None, 36)
            text = font.render("Shuffle", True, (255, 255, 255))
            screen.blit(text, (windowX - 140, 25))
            pygame.display.flip()

        # TODO: implement phase summary here to help with implementing skip if in next phase
        summarize_phase()
        print(ball_phase_summaries)
        current_phase += 1
        print('Current time',current_seconds,'end tiime',end_time)

    # All phases complete: cleanly quit pygame and exit the process
    try:
        pygame.quit()
    except Exception:
        pass
    # sys.exit()

# %%
if __name__ == "__main__":

    main()

# Make sure the folder exists
save_dir = "experiment_data"
os.makedirs(save_dir, exist_ok=True)

# Build timestamped filename
timestamp = strftime("%Y-%m-%d_%H-%M-%S")
filename = f"{timestamp}_session_summary.json"

# Full path inside experiment_data
filepath = os.path.join(save_dir, filename)

# Write the file
with open(filepath, "w") as f:
    json.dump(ball_phase_summaries, f, indent=2)

print("Saved to:", filepath)

print(current_seconds)

Settings loaded from settings.json
{'phase_1': {'experiment_widgets.duration': 5, 'experiment_widgets.background_color': '#04ff00', 'experiment_widgets.before_instructions': 'click to start - Press red ball to earn points', 'experiment_widgets.during_instructions': '', 'experiment_widgets.number_of_balls': 2, 'experiment_widgets.initial_speeds': [1.0, 1.0], 'experiment_widgets.radii': [60.0, 60.0], 'experiment_widgets.points_per_reinforcement': [1, 0], 'experiment_widgets.change_to_clicks': [1, 1], 'experiment_widgets.change_over_delay': [0.0, 0.0], 'experiment_widgets.base_colors': ['#ff0000', '#0000ff'], 'experiment_widgets.time_required': [0.0, 0.0], 'experiment_widgets.clicks_required': [1, 1], 'experiment_widgets.change_from_clicks': [1, 1], 'experiment_widgets.change_from_delay': [1.0, 1.0], 'experiment_widgets.block_score_until_time': [0.0, 0.0], 'experiment_widgets.block_score_until_clicks': [0, 0], 'experiment_widgets.zero_points_message': ['', ''], 'experiment_widgets.debug':

## How to access variables

1. ball_phase_summaries[current_phase-1]["ball_1"]["clicks"]
2. ball_phase_summaries[current_phase-1]["ball_1"]["valid_clicks"]
3. ball_phase_summaries[current_phase-1]["ball_1"]["score"]


In [ ]:
print("FREE CONTINGENCY CODE:\n---\n")
print(phase_options)
print("\n--- END ---")


## Just clicked only blue in phase 1, went to phase with blue background
## Correctly ended after

# When I click only red in phase 1 it does now go to t he phase with red background, YAY!